In [0]:
# Your storage details
storage_account = "insurancepocadls"
storage_key     = "2EN6t12bfbOxJ14PP+Ez6FmkPXu3EpC6/OPvs9WGWgaxy+by8EHsARB0uxJbWZSJqICJzcWugPrP+ASt7eAizw=="   # ← paste your actual key here

# Build the path to your bronze container
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net"

# Try reading customers file — key is passed directly inside the read
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .load(f"{bronze_path}/customers_500.csv")
)

display(df)

In [0]:
# Read all 4 files using the same method
customers = (spark.read.format("csv")
    .option("header", "true")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .load(f"{bronze_path}/customers_500.csv"))

policies = (spark.read.format("csv")
    .option("header", "true")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .load(f"{bronze_path}/policies_500.csv"))

claims = (spark.read.format("csv")
    .option("header", "true")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .load(f"{bronze_path}/claims_500.csv"))

providers = (spark.read.format("csv")
    .option("header", "true")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .load(f"{bronze_path}/providers_150.csv"))

# Print summary
print(f"✅ customers  → {customers.count()} rows  | {len(customers.columns)} columns")
print(f"✅ policies   → {policies.count()} rows  | {len(policies.columns)} columns")
print(f"✅ claims     → {claims.count()} rows  | {len(claims.columns)} columns")
print(f"✅ providers  → {providers.count()} rows | {len(providers.columns)} columns")

In [0]:
# Path where we want to save the Delta table
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net"

# Write customers as Delta table
(customers.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{silver_path}/customers")
)

print("✅ Customers saved as Delta table!")

In [0]:
# Write all 4 tables to silver as Delta
tables = {
    "customers" : customers,
    "policies"  : policies,
    "claims"    : claims,
    "providers" : providers,
}

for name, df in tables.items():
    (df.write
        .format("delta")
        .mode("overwrite")
        .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
        .save(f"{silver_path}/{name}")
    )
    print(f"✅ {name} saved!")

print("\n🎉 All 4 tables saved to Silver layer!")

In [0]:
from pyspark.sql import functions as F

silver_customers = (
    customers
    # Fix date columns
    .withColumn("date_of_birth", F.to_date("date_of_birth", "yyyy-MM-dd"))
    .withColumn("join_date",     F.to_date("join_date",     "yyyy-MM-dd"))

    # Calculate age in years
    .withColumn("age_years",
        (F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25)
        .cast("int"))

    # How many days have they been a customer
    .withColumn("tenure_days",
        F.datediff(F.current_date(), F.col("join_date")))

    # Age group label
    .withColumn("age_band",
        F.when(F.col("age_years") < 18,  "Under 18")
        .when(F.col("age_years") < 30,   "18-29")
        .when(F.col("age_years") < 45,   "30-44")
        .when(F.col("age_years") < 60,   "45-59")
        .otherwise("60+"))
)

display(silver_customers.select(
    "customer_id", "first_name", "date_of_birth",
    "age_years", "age_band", "tenure_days", "join_date"
).limit(5))

In [0]:
# Save cleaned customers to silver
(silver_customers.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{silver_path}/customers_cleaned")
)
print("✅ Silver customers saved!")

In [0]:
silver_policies = (
    policies
    # Fix date columns
    .withColumn("start_date", F.to_date("start_date", "yyyy-MM-dd"))
    .withColumn("end_date",   F.to_date("end_date",   "yyyy-MM-dd"))

    # Days remaining until policy expires
    .withColumn("days_to_expiry",
        F.datediff(F.col("end_date"), F.current_date()))

    # Is the policy already expired?
    .withColumn("is_expired",
        F.when(F.col("end_date") < F.current_date(), "Yes")
        .otherwise("No"))

    # How urgent is the renewal?
    .withColumn("renewal_urgency",
        F.when(F.col("days_to_expiry") < 0,    "Expired")
        .when(F.col("days_to_expiry") <= 30,   "Critical")
        .when(F.col("days_to_expiry") <= 90,   "High")
        .when(F.col("days_to_expiry") <= 180,  "Medium")
        .otherwise("Low"))
)

display(silver_policies.select(
    "policy_id", "customer_id", "policy_type",
    "start_date", "end_date",
    "days_to_expiry", "is_expired", "renewal_urgency"
).limit(5))

In [0]:
# Save cleaned policies
(silver_policies.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{silver_path}/policies_cleaned")
)
print("✅ Silver policies saved!")

In [0]:
silver_claims = (
    claims
    # Fix date column
    .withColumn("claim_date",     F.to_date("claim_date", "yyyy-MM-dd"))
    .withColumn("claim_amount",   F.col("claim_amount").cast("double"))
    .withColumn("approved_amount",F.col("approved_amount").cast("double"))

    # What % of claim was approved
    .withColumn("approval_rate",
        F.round(F.col("approved_amount") / F.col("claim_amount") * 100, 2))

    # Amount that was NOT approved
    .withColumn("rejected_amount",
        F.col("claim_amount") - F.col("approved_amount"))

    # Flag suspicious claims
    .withColumn("high_risk_flag",
        F.when(
            (F.col("status") == "Under Investigation") &
            (F.col("claim_amount") > 150000),
        "Yes").otherwise("No"))
)

display(silver_claims.select(
    "claim_id", "customer_id", "claim_amount",
    "approved_amount", "approval_rate",
    "rejected_amount", "status", "high_risk_flag"
).limit(5))

In [0]:
# Save cleaned claims
(silver_claims.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{silver_path}/claims_cleaned")
)
print("✅ Silver claims saved!")

In [0]:
silver_providers = (
    providers
    .withColumn("total_claims_filed",   F.col("total_claims_filed").cast("int"))
    .withColumn("average_claim_amount", F.col("average_claim_amount").cast("double"))
    .withColumn("rating",               F.col("rating").cast("double"))

    # Convert Yes/No to proper flag
    .withColumn("is_empanelled",
        F.when(F.upper(F.col("empanelled")) == "YES", "Yes")
        .otherwise("No"))

    # Rating label
    .withColumn("rating_band",
        F.when(F.col("rating") >= 4.5, "Excellent")
        .when(F.col("rating") >= 4.0, "Good")
        .when(F.col("rating") >= 3.0, "Average")
        .otherwise("Poor"))
)

display(silver_providers.select(
    "provider_id", "provider_name", "provider_type",
    "is_empanelled", "rating", "rating_band"
).limit(5))

In [0]:
# Save cleaned providers
(silver_providers.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{silver_path}/providers_cleaned")
)
print("✅ Silver providers saved!")
print("\n🎉 All Silver tables done!")
print("   ✅ customers_cleaned")
print("   ✅ policies_cleaned")
print("   ✅ claims_cleaned")
print("   ✅ providers_cleaned")

In [0]:
# Read all cleaned silver tables
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net"

s_customers = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{silver_path}/customers_cleaned")

s_policies = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{silver_path}/policies_cleaned")

s_claims = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{silver_path}/claims_cleaned")

s_providers = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{silver_path}/providers_cleaned")

print("✅ All silver tables loaded!")

In [0]:
# Step 1 — Summarise policies per customer
policy_summary = (
    s_policies
    .groupBy("customer_id")
    .agg(
        F.count("policy_id")                                          .alias("total_policies"),
        F.sum("sum_insured")                                          .alias("total_sum_insured"),
        F.sum("premium_amount")                                       .alias("total_premium"),
        F.sum(F.when(F.col("is_expired") == "No",  1).otherwise(0))  .alias("active_policies"),
        F.sum(F.when(F.col("is_expired") == "Yes", 1).otherwise(0))  .alias("expired_policies"),
        F.min("days_to_expiry")                                       .alias("nearest_expiry_days"),
        F.sum(F.when(F.col("renewal_urgency") == "Critical", 1)
               .otherwise(0))                                         .alias("critical_renewals"),
    )
)

# Step 2 — Summarise claims per customer
claims_summary = (
    s_claims
    .groupBy("customer_id")
    .agg(
        F.count("claim_id")                                                      .alias("total_claims"),
        F.sum("claim_amount")                                                    .alias("total_claimed"),
        F.sum("approved_amount")                                                 .alias("total_approved"),
        F.sum(F.when(F.col("status") == "Approved", 1).otherwise(0))            .alias("approved_claims"),
        F.sum(F.when(F.col("status") == "Rejected", 1).otherwise(0))            .alias("rejected_claims"),
        F.sum(F.when(F.col("status") == "Under Investigation", 1).otherwise(0)) .alias("flagged_claims"),
        F.sum(F.when(F.col("high_risk_flag") == "Yes", 1).otherwise(0))         .alias("high_risk_claims"),
    )
)

print("✅ Policy summary ready!")
print("✅ Claims summary ready!")

In [0]:
# Step 3 — Join everything together
customer_360 = (
    s_customers
    .join(policy_summary, on="customer_id", how="left")
    .join(claims_summary, on="customer_id", how="left")

    # Fill 0 for customers with no claims yet
    .fillna(0, subset=[
        "total_policies", "total_claims", "total_claimed",
        "total_approved", "approved_claims", "rejected_claims",
        "flagged_claims", "high_risk_claims", "critical_renewals"
    ])

    # Churn risk label
    .withColumn("churn_risk",
        F.when(
            (F.col("critical_renewals") > 0) &
            (F.col("rejected_claims")   > 0),
        "High")
        .when(
            (F.col("critical_renewals") > 0) |
            (F.col("expired_policies")  > 0),
        "Medium")
        .otherwise("Low"))
)

display(customer_360.select(
    "customer_id", "first_name", "last_name",
    "premium_tier", "total_policies", "active_policies",
    "total_claims", "high_risk_claims", "churn_risk"
).limit(5))

In [0]:
# Save Customer 360 to Gold layer
(customer_360.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{gold_path}/customer_360")
)
print("✅ Gold — Customer 360 saved!")

In [0]:
# Signal 1 — High value claims under investigation
signal_1 = (
    s_claims
    .filter(
        (F.col("status") == "Under Investigation") &
        (F.col("claim_amount") > 150000)
    )
    .withColumn("fraud_signal", F.lit("HIGH VALUE UNDER INVESTIGATION"))
    .withColumn("risk_level",   F.lit("Critical"))
    .select(
        "claim_id", "customer_id", "provider_id",
        "claim_date", "claim_amount", "approved_amount",
        "approval_rate", "status", "fraud_signal", "risk_level"
    )
)

print(f"🚨 Signal 1 — {signal_1.count()} suspicious claims found")
display(signal_1)

In [0]:
# Signal 2 — Claims at Non-Empanelled Providers
signal_2 = (
    s_claims
    .join(
        s_providers.select("provider_id", "is_empanelled", "provider_name"),
        on="provider_id", how="left"
    )
    .filter(F.col("is_empanelled") == "No")
    .withColumn("fraud_signal", F.lit("NON-EMPANELLED PROVIDER"))
    .withColumn("risk_level",
        F.when(F.col("claim_amount") > 100000, "High")
        .otherwise("Medium"))
    .select(
        "claim_id", "customer_id", "provider_id", "provider_name",
        "claim_date", "claim_amount", "approved_amount",
        "status", "fraud_signal", "risk_level"
    )
)

print(f"🚨 Signal 2 — {signal_2.count()} suspicious claims found")
display(signal_2.limit(5))

In [0]:
# Signal 3 — Same customer filing multiple claims within 30 days
from pyspark.sql.window import Window

# Look at each customer's claims ordered by date
window = Window.partitionBy("customer_id").orderBy("claim_date")

signal_3 = (
    s_claims
    # Find previous claim date for same customer
    .withColumn("prev_claim_date",
        F.lag("claim_date", 1).over(window))

    # Calculate days between claims
    .withColumn("days_since_last_claim",
        F.datediff(F.col("claim_date"), F.col("prev_claim_date")))

    # Keep only claims filed within 30 days of previous claim
    .filter(F.col("days_since_last_claim") <= 30)

    .withColumn("fraud_signal", F.lit("MULTIPLE CLAIMS WITHIN 30 DAYS"))
    .withColumn("risk_level",
        F.when(F.col("days_since_last_claim") <= 7,  "Critical")
        .when(F.col("days_since_last_claim") <= 14, "High")
        .otherwise("Medium"))
    .select(
        "claim_id", "customer_id", "claim_date",
        "prev_claim_date", "days_since_last_claim",
        "claim_amount", "status", "fraud_signal", "risk_level"
    )
)

print(f"🚨 Signal 3 — {signal_3.count()} suspicious claims found")
display(signal_3.limit(5))

In [0]:
# Signal 4 — Very low approval rate (less than 20% approved)
signal_4 = (
    s_claims
    .filter(
        (F.col("approval_rate") < 20) &
        (F.col("claim_amount") > 100000)
    )
    .withColumn("fraud_signal", F.lit("VERY LOW APPROVAL ON HIGH CLAIM"))
    .withColumn("risk_level",
        F.when(F.col("approval_rate") < 10, "Critical")
        .otherwise("High"))
    .select(
        "claim_id", "customer_id", "provider_id",
        "claim_date", "claim_amount", "approved_amount",
        "approval_rate", "status", "fraud_signal", "risk_level"
    )
)

print(f"🚨 Signal 4 — {signal_4.count()} suspicious claims found")
display(signal_4.limit(5))

In [0]:
# Combine all 4 signals into one Fraud Signals table
# First align columns across all signals
common_cols = [
    "claim_id", "customer_id", "provider_id",
    "claim_date", "claim_amount", "approved_amount",
    "status", "fraud_signal", "risk_level"
]

# Add missing columns to signals that don't have them
signal_1_aligned = signal_1.withColumn("provider_id", F.col("provider_id"))
signal_2_aligned = signal_2.withColumn("approved_amount", F.col("approved_amount"))
signal_3_aligned = (signal_3
    .withColumn("provider_id",      F.lit("Unknown"))
    .withColumn("approved_amount",  F.col("claim_amount") * 0))
signal_4_aligned = signal_4

# Union all signals
all_fraud_signals = (
    signal_1_aligned.select(common_cols)
    .union(signal_2_aligned.select(common_cols))
    .union(signal_3_aligned.select(common_cols))
    .union(signal_4_aligned.select(common_cols))
    .dropDuplicates(["claim_id", "fraud_signal"])
)

print(f"🚨 Total fraud signals found : {all_fraud_signals.count()}")
print(f"   Signal 1 - High Value Under Investigation : 63")
print(f"   Signal 2 - Non-Empanelled Provider        :  5")
print(f"   Signal 3 - Multiple Claims in 30 Days     :  5")
print(f"   Signal 4 - Very Low Approval Rate         :  5")

display(all_fraud_signals.orderBy("risk_level"))

In [0]:
# Save Fraud Signals to Gold layer
(all_fraud_signals.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{gold_path}/fraud_signals")
)
print("✅ Gold — Fraud Signals saved!")

In [0]:
# Join policies with claims to calculate loss ratio
claims_per_policy = (
    s_claims
    .groupBy("policy_id")
    .agg(
        F.count("claim_id")         .alias("total_claims"),
        F.sum("claim_amount")       .alias("total_claimed"),
        F.sum("approved_amount")    .alias("total_approved"),
        F.sum(F.when(F.col("high_risk_flag") == "Yes", 1)
               .otherwise(0))       .alias("high_risk_claims"),
    )
)

# Build Policy Renewal Risk table
policy_renewal = (
    s_policies
    .join(claims_per_policy, on="policy_id", how="left")

    # Fill 0 for policies with no claims
    .fillna(0, subset=[
        "total_claims", "total_claimed",
        "total_approved", "high_risk_claims"
    ])

    # Loss ratio = how much paid out vs premium collected
    .withColumn("loss_ratio",
        F.when(F.col("premium_amount") > 0,
            F.round(F.col("total_approved") / F.col("premium_amount"), 2)
        ).otherwise(F.lit(0.0)))

    # Profitability label
    .withColumn("profitability",
        F.when(F.col("loss_ratio") > 1.5, "Loss Making")
        .when(F.col("loss_ratio") > 1.0,  "Unprofitable")
        .when(F.col("loss_ratio") > 0.7,  "Marginal")
        .otherwise("Profitable"))

    # What action should the business take?
    .withColumn("action_needed",
        F.when(F.col("is_expired")       == "Yes",      "Winback Campaign")
        .when(F.col("renewal_urgency")   == "Critical",  "Call Immediately")
        .when(
            (F.col("renewal_urgency")    == "High") &
            (F.col("loss_ratio")         >  1.0),        "Reprice Before Renewal")
        .when(F.col("renewal_urgency")   == "High",      "Schedule a Call")
        .when(F.col("renewal_urgency")   == "Medium",    "Send Email Reminder")
        .otherwise("Standard Renewal Flow"))

    .select(
        "policy_id", "customer_id", "policy_type",
        "premium_amount", "sum_insured",
        "days_to_expiry", "renewal_urgency", "is_expired",
        "total_claims", "total_approved", "loss_ratio",
        "profitability", "high_risk_claims", "action_needed"
    )
)

display(policy_renewal.orderBy("days_to_expiry").limit(5))

In [0]:
# Save Policy Renewal Risk to Gold layer
(policy_renewal.write
    .format("delta")
    .mode("overwrite")
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    .save(f"{gold_path}/policy_renewal_risk")
)
print("✅ Gold — Policy Renewal Risk saved!")

In [0]:
# Final Summary — Everything we built today
print("=" * 55)
print("   INSURANCE POD — PROJECT SUMMARY")
print("=" * 55)

print("\n📂 SILVER LAYER (Cleaned Data)")
print(f"   customers_cleaned  → {s_customers.count():,} rows")
print(f"   policies_cleaned   → {s_policies.count():,} rows")
print(f"   claims_cleaned     → {s_claims.count():,} rows")
print(f"   providers_cleaned  → {s_providers.count():,} rows")

print("\n🥇 GOLD LAYER (Business Reports)")

c360 = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{gold_path}/customer_360")

fraud = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{gold_path}/fraud_signals")

renewal = spark.read.format("delta") \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .load(f"{gold_path}/policy_renewal_risk")

print(f"   customer_360       → {c360.count():,} customer profiles")
print(f"   fraud_signals      → {fraud.count():,} suspicious claims")
print(f"   policy_renewal     → {renewal.count():,} policies analysed")

print("\n🚨 KEY BUSINESS FINDINGS")
print(f"   High churn risk customers  → {c360.filter(F.col('churn_risk')=='High').count()}")
print(f"   Critical renewals          → {renewal.filter(F.col('renewal_urgency')=='Critical').count()}")
print(f"   Expired policies           → {renewal.filter(F.col('is_expired')=='Yes').count()}")
print(f"   Loss making policies       → {renewal.filter(F.col('profitability')=='Loss Making').count()}")
print(f"   Critical fraud signals     → {fraud.filter(F.col('risk_level')=='Critical').count()}")
print("=" * 55)
print("   ✅ Pipeline Complete!")
print("=" * 55)